<a href="https://cognitiveclass.ai"><img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DL0101EN-SkillsNetwork/images/IDSN-logo.png" width="400"> </a>

# Convolutional Neural Networks with Keras

Estimated time needed **30** mins


In this lab, we will learn how to use the Keras library to build convolutional neural networks. We will also use the popular MNIST dataset and we will compare our results to using a conventional neural network.


## Objectives for this Notebook    
* How to use the Keras library to build convolutional neural networks
* Convolutional neural network with one set of convolutional and pooling layers
* Convolutional neural network with two sets of convolutional and pooling layers



## Table of Contents

<div class="alert alert-block alert-info" style="margin-top: 20px">

<font size = 3>
      
1. <a href="#Import-Keras-and-Packages">Import Keras and Packages</a>   
2. <a href="#Convolutional-Neural-Network-with-One-Set-of-Convolutional-and-Pooling-Layers">Convolutional Neural Network with One Set of Convolutional and Pooling Layers</a>  
3. <a href="#Convolutional-Neural-Network-with-Two-Sets-of-Convolutional-and-Pooling-Layers">Convolutional Neural Network with Two Sets of Convolutional and Pooling Layers</a>  

</font>
</div>


### Install the necessary libraries


Let's start by installing the keras libraries and the packages that we would need to build a neural network.


In [1]:
# All Libraries required for this lab are listed below. The libraries need to be installed on Skills Network Labs. 
# If you run this notebook on a different environment, e.g. your desktop, you may want to install these.
#!pip install numpy==2.0.2
#!pip install pandas==2.2.2
#!pip install tensorflow_cpu==2.18.0
#!pip install matplotlib==3.9.2

# Installing the libraries, if not installed. Run the following lines to install the required libraries for this lab:
# Terminal>   pip install numpy
# Terminal>   pip install pandas
# Terminal>   pip install tensorflow_cpu
# Terminal>   pip install matplotlib
print("==== All required libraries should be installed =====")


==== All required libraries should be installed =====


#### Suppress the tensorflow warning messages
We use the following code to  suppress the warning messages due to use of CPU architechture for tensoflow.
You may want to **comment out** these lines if you are using the GPU architechture


In [2]:
import os
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

## Import Keras and Packages


In [ ]:
import keras
from keras.models import Sequential  # import the Sequential model from Keras
from keras.layers import Dense # import the Dense layer from Keras
from keras.layers import Input # import the Input layer from Keras
from keras.utils import to_categorical # import the to_categorical function from Keras, which is used for one-hot encoding of labels

When working with convolutional neural networks in particular, we will need additional packages.


In [4]:
from keras.layers import Conv2D # to add convolutional layers
from keras.layers import MaxPooling2D # to add pooling layers
from keras.layers import Flatten # to flatten data for fully connected layers

## Convolutional Neural Network with One Set of Convolutional and Pooling Layers


In [ ]:
## import data
from keras.datasets import mnist # This line of code imports the MNIST dataset from the Keras library.
#  The MNIST dataset is a collection of 70,000 handwritten digit images, commonly used for
#   training and testing machine learning models in the field of computer vision.
#   The dataset is divided into a training set of 60,000 images and a test set of 10,000 images, 
#   with each image being a 28x28 pixel grayscale image representing a digit from 0 to 9.

## Load data
(X_train, y_train), (X_test, y_test) = mnist.load_data() # This line of code loads the MNIST dataset and splits it into training and test sets.
#  The load_data() function returns two tuples: (X_train, y_train) and (X_test, y_test). 
#  X_train and X_test contain the image data for the training and test sets, respectively, 
#  while y_train and y_test contain the corresponding labels for each image. 

## Reshape to be [samples][pixels][width][height]
X_train = X_train.reshape(X_train.shape[0], 28, 28, 1).astype('float32') 
# This line of code reshapes the training data (X_train) to have a shape of (number of samples, 28, 28, 1) and converts the data type to float32.
#   The reshape() function is used to change the shape of the data, and in this case, it adds
#   an additional dimension to represent the number of channels (1 for grayscale images).
X_test = X_test.reshape(X_test.shape[0], 28, 28, 1).astype('float32') 
# This line of code reshapes the test data (X_test) in the same way as the training data,
#   to have a shape of (number of samples, 28, 28, 1) and converts the data type to float32. 

Let's normalize the pixel values to be between 0 and 1


In [6]:
X_train = X_train / 255 # normalize training data
#  This line of code normalizes the training data (X_train) by dividing each pixel value by 255.
#  Normalization is a common preprocessing step in machine learning that scales the pixel values to
#  a range of 0 to 1. This is done to improve the performance of the model during training

X_test = X_test / 255 # normalize test data
#  This line of code normalizes the test data (X_test) by dividing each pixel value by 255.
#  Normalization is a common preprocessing step in machine learning that scales the pixel values to
#  a range of 0 to 1. This is done to improve the performance of the model during training


Next, let's convert the target variable into binary categories


In [ ]:
## To Convert labels to one-hot encoding: 
y_train = to_categorical(y_train) 
# This line of code converts the training labels (y_train) into a one-hot encoded format using
#  the to_categorical() function from Keras. One-hot encoding is a technique used to represent 
#  categorical variables as binary vectors. In this case, each label in y_train is converted into
#  a binary vector of length 10 (since there are 10 classes in the MNIST dataset), 
#  where the index corresponding to the class label is set to 1 and all other indices are set to 0.

y_test = to_categorical(y_test)
# This line of code converts the test labels (y_test) into a one-hot encoded format using
#  the to_categorical() function from Keras, similar to how the training labels were converted.
#  Each label in y_test is transformed into a binary vector of length 10, where the index 
#  corresponding to the class label is set to 1 and all other indices are set to 0.

num_classes = y_test.shape[1] # number of categories
# This line of code calculates the number of classes (categories) in the dataset by accessing the shape of y_test.
#  The shape of y_test is (number of samples, number of classes), so y_test.shape[1] gives us the number 
#  of classes, which is stored in the variable num_classes. In the case of the MNIST dataset, num_classes
#  will be equal to 10, since there are 10 different digit classes (0-9). 

Next, let's define a function that creates our model. Let's start with one set of convolutional and pooling layers.


In [8]:
def convolutional_model():
    
    ## To create model, we will use the Sequential API of Keras, which allows us to build a model layer by layer.
    model = Sequential() # This line of code initializes a new Sequential model in Keras.
     # The Sequential model is a linear stack of layers, where you can add layers one after the
     #  other to create a neural network architecture. In this case, we will be adding convolutional layers,
     #  pooling layers, and fully connected layers to build our convolutional neural network (CNN) for image classification. 
    model.add(Input(shape=(28, 28, 1))) # This line of code adds an Input layer to the model, 
     #  specifying the shape of the input data. The Input layer is the first layer of the model and
     #  defines the shape of the input data. In this case, the input shape is (28, 28, 1), which gives 
     #  us 28x28 pixel grayscale images (1 channel). This layer does not perform any computations but
     #  serves as a placeholder for the input data. 
    model.add(Conv2D(16, (5, 5), strides=(1, 1), activation='relu')) # This line of code adds a 
     #  2D convolutional layer to the model with 16 filters, a kernel size of (5, 5), strides 
     #  of (1, 1), and ReLU activation function. The Conv2D layer applies convolutional operations
     #  to the input data, which helps the model learn spatial hierarch ies and features from the images.
     #  The number of filters (16) determines how many different features the layer will learn,
     #  while the kernel size (5, 5) specifies the size of the convolutional filters. The strides (1, 1) 
     #  indicate how the filters will move across the input data, and the activation function (ReLU) introduces
     #  non-linearity to the model, allowing it to learn more complex patterns.
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2))) # This line of code adds a MaxPooling2D
      #  layer to the model with a pool size of (2, 2) and strides of (2, 2). The MaxPooling2D layer
      #  performs downsampling on the input data by taking the maximum value from each pool of the 
      #  specified size. In this case, the pool size is (2, 2), which means that the layer will take
      #  the maximum value from each 2x2 region of the input data. The strides (2, 2) indicate how the
      #  pooling operation will move across the input data, effectively reducing the spatial dimensions 
      #  of the feature maps by a factor of 2. This helps to reduce the computational complexity of the 
      #  model and can also help to prevent overfitting by providing a form of spatial invariance.
    
    model.add(Flatten())  # This line of code adds a Flatten layer to the model, which is used to flatten
       # the output from the previous layers into a 1D vector. The Flatten layer takes the multi-dimensional
       # output from the convolutional and pooling layers and reshapes it into a one-dimensional vector that
       # can be fed into the fully connected layers. This is necessary because the Dense layers that follow
       # expect a 1D input. The Flatten layer does not perform any computations; it simply reshapes the 
       # data to prepare it for the next layers in the model.
    model.add(Dense(100, activation='relu'))  # This line of code adds a Dense layer to the model with 100 units
       # and ReLU activation function. The Dense layer is a fully connected layer that connects every neuron in 
       # the previous layer to every neuron in the current layer. In this case, the Dense layer has 100 units,
       # which means it will learn 100 different features from the input data. The activation 
       # function ReLU (Rectified Linear Unit) introduces non-linearity to the model, allowing it to learn more
       #  complex patterns in the data. The Dense layer will take the flattened output from the previous layers and
       #  learn to classify the images based on the features it has extracted. 
    model.add(Dense(num_classes, activation='softmax'))  # This line of code adds a Dense layer to the model with
       #  a number of units equal to num_classes and a softmax activation function. The number of units in this 
       #  layer corresponds to the number of classes in the dataset (10 for MNIST), and the softmax activation 
       #  function is used for multi-class classification problems. The softmax function converts the output of 
       #  the layer into a probability distribution over the classes, where the output values represent the 
       #  probability of each class being the correct label for the input image. This layer serves as
       #  the output layer of the model, and it will produce the final predictions for the classification task.
    
    ## To compile model:
    model.compile(optimizer='adam', loss='categorical_crossentropy',  metrics=['accuracy'])  # This line of code 
     #  compiles the model by specifying the optimizer, loss function, and evaluation metrics. The optimizer 'adam'
     #  is an adaptive learning rate optimization algorithm that is commonly used for training neural networks. 
     #  The loss function 'categorical_crossentropy' is used for multi-class classification problems, and it 
     #  measures the difference between the predicted probability distribution and the true distribution of 
     #  the labels. The metrics=['accuracy'] argument specifies that we want to evaluate the model's performance
     #  using accuracy during training and testing. Compiling the model prepares it for training by configuring
     #  the learning process based on the specified parameters.
    return model  # This line of code returns the compiled model, which can then be used for training and making predictions. 


Finally, let's call the function to create the model, and then let's train it and evaluate it.


In [9]:
## To build the model:
model = convolutional_model()  # This line of code calls the convolutional_model() function to build 
 #  and compile the model. The function creates a convolutional neural network architecture and compiles
 #  it with the specified optimizer, loss function, and evaluation metrics. The resulting model is 
 #  stored in the variable 'model', which can then be used for training and making predictions on 
 #  the MNIST dataset.

## To fit the model:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=200, verbose=2)
 # This line of code trains the model using the fit() method. The training data (X_train and y_train) is
 #  passed as input, along with the validation data (X_test and y_test) to evaluate the model's performance
 #  on unseen data during training. The number of epochs is set to 10, which means the model will iterate
 #  through the entire training dataset 10 times. The batch size is set to 200, which means that the 
 #  model will update its weights after processing every 200 samples. The verbose parameter is set to 2, 
 #  which provides a more detailed output during training, showing the progress of each epoch and the 
 #  corresponding loss and accuracy metrics.

## To evaluate the model:
scores = model.evaluate(X_test, y_test, verbose=0)  # This line of code evaluates the performance of the 
#  trained model on the test data (X_test and y_test) using the evaluate() method. The method returns a 
#   list of metrics, where the first element (scores[0]) is the loss value and the second element (scores[1])
#   is the accuracy of the model on the test data. The verbose parameter is set to 0, which means that no 
#   output will be printed during the evaluation process. 
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))  # This line of code prints the 
#   accuracy and error rate of the model on the test data. The accuracy is obtained from scores[1], 
#   which is the second element of the list returned by the evaluate() method. The error rate is 
#   calculated as 100 minus the accuracy multiplied by 100, giving us the percentage of incorrect 
#   predictions made by the model on the test data. 

Epoch 1/10
300/300 - 17s - 57ms/step - accuracy: 0.9077 - loss: 0.3201 - val_accuracy: 0.9639 - val_loss: 0.1281
Epoch 2/10
300/300 - 14s - 45ms/step - accuracy: 0.9698 - loss: 0.1025 - val_accuracy: 0.9770 - val_loss: 0.0730
Epoch 3/10
300/300 - 14s - 47ms/step - accuracy: 0.9807 - loss: 0.0648 - val_accuracy: 0.9818 - val_loss: 0.0563
Epoch 4/10
300/300 - 16s - 54ms/step - accuracy: 0.9849 - loss: 0.0485 - val_accuracy: 0.9832 - val_loss: 0.0510
Epoch 5/10
300/300 - 15s - 49ms/step - accuracy: 0.9879 - loss: 0.0396 - val_accuracy: 0.9835 - val_loss: 0.0474
Epoch 6/10
300/300 - 15s - 49ms/step - accuracy: 0.9901 - loss: 0.0326 - val_accuracy: 0.9858 - val_loss: 0.0446
Epoch 7/10
300/300 - 14s - 47ms/step - accuracy: 0.9916 - loss: 0.0268 - val_accuracy: 0.9865 - val_loss: 0.0413
Epoch 8/10
300/300 - 15s - 51ms/step - accuracy: 0.9936 - loss: 0.0221 - val_accuracy: 0.9878 - val_loss: 0.0364
Epoch 9/10
300/300 - 14s - 47ms/step - accuracy: 0.9941 - loss: 0.0192 - val_accuracy: 0.9882 - 

------------------------------------------


## Convolutional Neural Network with Two Sets of Convolutional and Pooling Layers


Let's redefine our convolutional model so that it has two convolutional and pooling layers instead of just one layer of each.


In [10]:
def convolutional_model():
    
    ## To create model:
    model = Sequential()  # This line of code initializes a new Sequential model in Keras.
     #  The Sequential model is a linear stack of layers, where you can add layers one after the
     #  other to create a neural network architecture. In this case, we will be adding convolutional layers,
     #  pooling layers, and fully connected layers to build our convolutional neural network (CNN) for image
     #  classification. 
    model.add(Input(shape=(28, 28, 1)))  # This line of code adds an Input layer to the model, specifying the
     #  shape of the input data. The Input layer is the first layer of the model and defines the shape of the 
     #  input data. In this case, the input shape is (28, 28, 1), which gives us 28x28 pixel grayscale 
     #  images (1 channel). This layer does not perform any computations but serves as a placeholder for
     #  the input data. 
    model.add(Conv2D(16, (5, 5), activation='relu'))  # This line of code adds a 2D convolutional layer to
     #  the model with 16 filters, a kernel size of (5, 5), and ReLU activation function. The Conv2D layer applies
     #  convolutional operations to the input data, which helps the model learn spatial hierarchies and 
     #  features from the images. The number of filters (16) determines how many different features the layer
     #  will learn, while the kernel size (5, 5) specifies the size of the convolutional filters. The 
     #  activation function ReLU (Rectified Linear Unit) introduces non-linearity to the model, allowing
     #  it to learn more complex patterns. 
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))   # This line of code adds a MaxPooling2D layer
     #  to the model with a pool size of (2, 2) and strides of (2, 2). The MaxPooling2D layer performs 
     #  downsampling on the input data by taking the maximum value from each pool of the specified size.
     #  In this case, the pool size is (2, 2), which means that the layer will take the maximum value from
     #  each 2x2 region of the input data. The strides (2, 2) indicate how the pooling operation will move
     #  across the input data, effectively reducing the spatial dimensions of the feature maps by a factor 
     #  of 2. This helps to reduce the computational complexity of the model and can also help to prevent 
     #  overfitting by providing a form of spatial invariance. 
    
    model.add(Conv2D(8, (2, 2), activation='relu'))  # This line of code adds another 2D convolutional layer
     #  to the model with 8 filters, a kernel size of (2, 2), and ReLU activation function. This layer will learn
     #  additional features from the input data after the first convolutional and pooling layers. The
     #  number of filters (8) is smaller than the previous layer, which can help to reduce the number of
     #  parameters in the model and prevent overfitting. The kernel size (2, 2) allows the layer to learn
     #  smaller features from the input data, and the ReLU activation function continues to introduce 
     #  non-linearity to the model, enabling it to learn more complex patterns in the data. 
    model.add(MaxPooling2D(pool_size=(2, 2), strides=(2, 2)))  # This line of code adds another MaxPooling2D 
    #  layer to the model with the same pool size and strides as before. This layer will further reduce 
    #  the spatial dimensions of the feature maps, allowing the model to focus on the most important 
    #  features and reducing the computational complexity for the subsequent layers. The pooling operation
    #  will help to make the model more robust to variations in the input data and can also help to prevent
    #  overfitting by providing a form of spatial invariance.
    
    model.add(Flatten())  # This line of code adds a Flatten layer to the model, which is used to flatten
     #  the output from the previous layers into a 1D vector. The Flatten layer takes the multi-dimensional
     #  output from the convolutional and pooling layers and reshapes it into a 1D vector for further processing. 
    model.add(Dense(100, activation='relu'))  # This line of code adds a Dense layer to the model with 100 units
     #  and ReLU activation function. The Dense layer connects every neuron in the previous layer to every neuron
     #  in the current layer, and it will learn to classify the images based on the features extracted 
     #  by the convolutional layers. The number of units (100) determines how many different features the layer 
     #  will learn, and the ReLU activation function introduces non-linearity to the model, allowing it to
     #  learn more complex patterns in the data. 
    model.add(Dense(num_classes, activation='softmax'))  # This line of code adds a Dense layer to the model
    #  with a number of units equal to num_classes and a softmax activation function. The softmax activation
    #  function is used for multi-class classification problems, and it converts the output of the layer into
    #  a probability distribution over the classes. The output values represent the probability of each class
    #  being the correct label for the input image. This layer serves as the output layer of the model, and it
    #  will produce the final predictions for the classification task. 
    
    ## To Compile model:
    model.compile(optimizer='adam', loss='categorical_crossentropy',  metrics=['accuracy'])  # This line of code
    #  compiles the model by specifying the optimizer, loss function, and evaluation metrics. 
    #  The optimizer 'adam' is an adaptive learning rate optimization algorithm that is commonly used for
    #  training neural networks. The loss function 'categorical_crossentropy' is used for multi-class 
    #  classification problems, and it measures the difference between the predicted probability distribution
    #  and the true distribution of the labels. The metrics=['accuracy'] argument specifies that we want to
    #  evaluate the model's performance using accuracy during training and testing. Compiling the model 
    #  prepares it for training by configuring the learning process based on the specified parameters. 
    return model

Now, let's call the function to create our new convolutional neural network, and then let's train it and evaluate it.


In [11]:
## To build the model:
model = convolutional_model()

## To fit the model:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=200, verbose=2)

## To evaluate the model:
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))

Epoch 1/10
300/300 - 22s - 73ms/step - accuracy: 0.8553 - loss: 0.4781 - val_accuracy: 0.9598 - val_loss: 0.1413
Epoch 2/10
300/300 - 19s - 64ms/step - accuracy: 0.9640 - loss: 0.1196 - val_accuracy: 0.9728 - val_loss: 0.0863
Epoch 3/10
300/300 - 19s - 64ms/step - accuracy: 0.9744 - loss: 0.0853 - val_accuracy: 0.9767 - val_loss: 0.0751
Epoch 4/10
300/300 - 22s - 72ms/step - accuracy: 0.9782 - loss: 0.0700 - val_accuracy: 0.9811 - val_loss: 0.0588
Epoch 5/10
300/300 - 20s - 65ms/step - accuracy: 0.9818 - loss: 0.0606 - val_accuracy: 0.9826 - val_loss: 0.0543
Epoch 6/10
300/300 - 20s - 66ms/step - accuracy: 0.9836 - loss: 0.0526 - val_accuracy: 0.9850 - val_loss: 0.0457
Epoch 7/10
300/300 - 20s - 66ms/step - accuracy: 0.9861 - loss: 0.0461 - val_accuracy: 0.9865 - val_loss: 0.0415
Epoch 8/10
300/300 - 20s - 66ms/step - accuracy: 0.9868 - loss: 0.0420 - val_accuracy: 0.9868 - val_loss: 0.0410
Epoch 9/10
300/300 - 19s - 64ms/step - accuracy: 0.9884 - loss: 0.0374 - val_accuracy: 0.9856 - 

<h3>Practice Exercise 1</h3>


Let's see how batch size affects the time required and accuracy of the model training. 
For this, you can try to change batch_size to 1024 and check it's effect on accuracy


In [12]:
# Write your answer here - Practice Exercise 1
## To build the model :
model = convolutional_model()  # This line of code calls the convolutional_model() function to build
#   and compile the model. The function creates a convolutional neural network architecture and 
#   compiles it with the specified optimizer, loss function, and evaluation metrics. The resulting model 
#   is stored in the variable 'model', which can then be used for training and making predictions on the
#   MNIST dataset. 

## To fit the model:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=1024, verbose=2) # This line of 
#  code trains the model using the fit() method. The training data (X_train and y_train) is 
#  passed as input, along with the validation data (X_test and y_test) to evaluate the model's performance
#  on unseen data during training. The number of epochs is set to 10, which means the model will iterate
#   through the entire training dataset 10 times. The batch size is set to 1024, which means that the model 
#   will update its weights after processing every 1024 samples. 

## To evaluate the model:
scores = model.evaluate(X_test, y_test, verbose=0)  # This line of code evaluates the performance of the trained
    #  model on the test data (X_test and y_test) using the evaluate() method. The method returns a list of 
    #  metrics, where the first element (scores[0]) is the loss value and the second element (scores[1]) is
    #  the accuracy of the model on the test data. The verbose parameter is set to 0, which means that no output
    #  will be printed during the evaluation process. 
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))  # This line of code prints the accuracy
    #  and error rate of the model on the test data. The accuracy is obtained from scores[1], which is the 
    #  second element of the list returned by the evaluate() method. The error rate is calculated as 100 minus
    #  the accuracy multiplied by 100, giving us the percentage of incorrect predictions made by the model on
    #  the test data. 


Epoch 1/10
59/59 - 18s - 303ms/step - accuracy: 0.6741 - loss: 1.1858 - val_accuracy: 0.8797 - val_loss: 0.4243
Epoch 2/10
59/59 - 16s - 275ms/step - accuracy: 0.9016 - loss: 0.3399 - val_accuracy: 0.9224 - val_loss: 0.2579
Epoch 3/10
59/59 - 14s - 245ms/step - accuracy: 0.9325 - loss: 0.2320 - val_accuracy: 0.9461 - val_loss: 0.1850
Epoch 4/10
59/59 - 15s - 258ms/step - accuracy: 0.9487 - loss: 0.1779 - val_accuracy: 0.9588 - val_loss: 0.1396
Epoch 5/10
59/59 - 19s - 323ms/step - accuracy: 0.9571 - loss: 0.1446 - val_accuracy: 0.9669 - val_loss: 0.1150
Epoch 6/10
59/59 - 14s - 232ms/step - accuracy: 0.9620 - loss: 0.1267 - val_accuracy: 0.9706 - val_loss: 0.1030
Epoch 7/10
59/59 - 14s - 237ms/step - accuracy: 0.9671 - loss: 0.1113 - val_accuracy: 0.9713 - val_loss: 0.0943
Epoch 8/10
59/59 - 14s - 244ms/step - accuracy: 0.9705 - loss: 0.0991 - val_accuracy: 0.9722 - val_loss: 0.0846
Epoch 9/10
59/59 - 15s - 251ms/step - accuracy: 0.9737 - loss: 0.0891 - val_accuracy: 0.9772 - val_loss:

Double-click <b>here</b> for the solution.

<!-- Your answer is below:
# build the model
model = convolutional_model()

# fit the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=10, batch_size=1024, verbose=2)

# evaluate the model
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))
-->


<h3>Practice Exercise 2</h3>


Now, let's see how number of epochs  affect the time required and accuracy of the model training. 
For this, you can keep the batch_size=1024 and epochs=25 and check it's effect on accuracy


In [13]:
# Write your answer here - Practice Ex 2
## To build the model:
model = convolutional_model()  # This line of code calls the convolutional_model() function to build and
#   compile the model. The function creates a convolutional neural network architecture and compiles it 
#   with the specified optimizer, loss function, and evaluation metrics. The resulting model is stored
#   in the variable 'model', which can then be used for training and making predictions on the MNIST dataset. 

## To fit the model:
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=25, batch_size=1024, verbose=2)  # This line 
#   of code trains the model using the fit() method. The training data (X_train and y_train) is passed as input,
#   along with the validation data (X_test and y_test) to evaluate the model's performance on unseen data 
#   during training. The number of epochs is set to 25, which means the model will iterate through the 
#   entire training dataset 25 times. The batch size is set to 1024, which means that the model will 
#   update its weights after processing every 1024 samples. The verbose parameter is set to 2, which provides
#   a more detailed output during training, showing the progress of each epoch and the corresponding loss and
#   accuracy metrics.

## To evaluate the model:
scores = model.evaluate(X_test, y_test, verbose=0)  # This line of code evaluates the performance of the trained
    #  model on the test data (X_test and y_test) using the evaluate() method. The method returns a list of 
    #  metrics, where the first element (scores[0]) is the loss value and the second element (scores[1]) is
    #  the accuracy of the model on the test data. The verbose parameter is set to 0, which means that no output
    #  will be printed during the evaluation process. 
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))  # This line of code prints the accuracy
    #   and error rate of the model on the test data. The accuracy is obtained from scores[1], which is the second
    #   element of the list returned by the evaluate() method. The error rate is calculated as 100 minus the 
    #   accuracy multiplied by 100, giving us the percentage of incorrect predictions made by the model on the test data.




Epoch 1/25
59/59 - 12s - 206ms/step - accuracy: 0.6548 - loss: 1.2892 - val_accuracy: 0.8865 - val_loss: 0.4070
Epoch 2/25
59/59 - 10s - 167ms/step - accuracy: 0.9156 - loss: 0.2899 - val_accuracy: 0.9446 - val_loss: 0.1932
Epoch 3/25
59/59 - 9s - 161ms/step - accuracy: 0.9495 - loss: 0.1715 - val_accuracy: 0.9634 - val_loss: 0.1295
Epoch 4/25
59/59 - 10s - 161ms/step - accuracy: 0.9625 - loss: 0.1263 - val_accuracy: 0.9696 - val_loss: 0.1004
Epoch 5/25
59/59 - 10s - 163ms/step - accuracy: 0.9685 - loss: 0.1054 - val_accuracy: 0.9738 - val_loss: 0.0859
Epoch 6/25
59/59 - 10s - 162ms/step - accuracy: 0.9726 - loss: 0.0917 - val_accuracy: 0.9771 - val_loss: 0.0759
Epoch 7/25
59/59 - 10s - 163ms/step - accuracy: 0.9754 - loss: 0.0821 - val_accuracy: 0.9805 - val_loss: 0.0655
Epoch 8/25
59/59 - 11s - 180ms/step - accuracy: 0.9773 - loss: 0.0729 - val_accuracy: 0.9818 - val_loss: 0.0602
Epoch 9/25
59/59 - 11s - 180ms/step - accuracy: 0.9795 - loss: 0.0670 - val_accuracy: 0.9826 - val_loss: 

Double-click <b>here</b> for the solution.

<!-- Your answer is below:
# build the model
model = convolutional_model()

# fit the model
model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=25, batch_size=1024, verbose=2)

# evaluate the model
scores = model.evaluate(X_test, y_test, verbose=0)
print("Accuracy: {} \n Error: {}".format(scores[1], 100-scores[1]*100))


    -->


### Thank you for completing this lab!

This notebook was created by [Alex Aklson](https://www.linkedin.com/in/aklson/). I hope you found this lab interesting and educational. Feel free to contact me if you have any questions!


<!--
## Change Log

|  Date (YYYY-MM-DD) |  Version | Changed By  |  Change Description |
|---|---|---|---|
| 2024-11-20  | 3.0  | Aman  |  Updated the library versions to current |
| 2020-09-21  | 2.0  | Srishti  |  Migrated Lab to Markdown and added to course repo in GitLab |



<hr>

## <h3 align="center"> © IBM Corporation. All rights reserved. <h3/>


## <h3 align="center"> &#169; IBM Corporation. All rights reserved. <h3/>

